# V0 — Boundary-prior 18S cropping pipeline

End-to-end walkthrough on a single 2048×2048 px ROI (DB_top100_018, Fibroblast × Melanoma).

**Goal:** Apply a segmentation-agnostic cut to the 18S channel that dims predicted heterotypic boundaries, then run Cellpose-SAM on the modified channel. The cut is built from transcripts + morphology only — no prior segmentation is consumed.

Each section maps to one of the nine pipeline stages in `lib.py`.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
import numpy as np, pandas as pd, tifffile, matplotlib.pyplot as plt
from skimage.segmentation import find_boundaries
from scipy.ndimage import gaussian_filter
import lib
print(f"V0 constants:")
print(f"  LINEAGES (K={lib.K}): {lib.LINEAGES}")
print(f"  pixel size: {lib.PIXEL_UM} µm/px")
print(f"  σ_diffuse: {lib.SIGMA_UM} µm = {lib.SIGMA_DIFFUSE_PX:.3f} px")
print(f"  Dirichlet α = {lib.ALPHA}")
print(f"  grey-abstain N_min = {lib.N_MIN}")
print(f"  tanh sharpening K_sharp = {lib.K_SHARP}")
print(f"  σ_cut = {lib.SIGMA_CUT_PX} px ≈ {lib.SIGMA_CUT_PX * lib.PIXEL_UM:.2f} µm")
print(f"  depth_max = {lib.DEPTH_MAX}")
print(f"  qv ≥ {lib.QV_MIN}")
print(f"  CP-SAM: {lib.CPSAM_KW}")

## Stage 1 — ROI definition

2048×2048 px crop centred on cell 018 (clamped to fit inside the WSI's right edge). The crop is 435.2 µm on a side at 0.2125 µm/px.

In [ ]:
BENCHMARK_ID = "DB_top100_018"
CROP_PX = 2048

bench = pd.read_parquet(lib.DATA / "benchmark_doublets.parquet")
row = bench[bench["benchmark_id"] == BENCHMARK_ID].iloc[0]
cx_um, cy_um = float(row["x_um"]), float(row["y_um"])
lineage_pair = row["lineage_pair"]

# WSI dimensions
import tifffile as tf_mod
with tf_mod.TiffFile(lib.DAPI_TIF) as tf:
    wsi_H, wsi_W = tf.series[0].shape[-2:]

y0, y1, x0, x1 = lib.make_roi_bbox_centred(cx_um, cy_um, CROP_PX, wsi_H, wsi_W)
print(f"{BENCHMARK_ID} ({lineage_pair})")
print(f"  centroid: ({cx_um:.1f}, {cy_um:.1f}) µm")
print(f"  ROI bbox (WSI px):  y=[{y0}, {y1}]  x=[{x0}, {x1}]")
print(f"  ROI side: {y1-y0} × {x1-x0} px = {(y1-y0)*lib.PIXEL_UM:.1f} × {(x1-x0)*lib.PIXEL_UM:.1f} µm")

In [ ]:
dapi, s18 = lib.load_morphology(y0, y1, x0, x1)
H, W = s18.shape
print(f"DAPI: {dapi.shape} {dapi.dtype} range=[{dapi.min()}, {dapi.max()}]")
print(f"18S : {s18.shape} {s18.dtype} range=[{s18.min()}, {s18.max()}]")

fig, ax = plt.subplots(1, 2, figsize=(11, 5.5))
ax[0].imshow(np.log1p(dapi), cmap="gray"); ax[0].set_title("DAPI (log1p)")
ax[1].imshow(np.log1p(s18),  cmap="gray"); ax[1].set_title("18S (log1p)")
for a in ax: a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()

## Stage 2 — Gene → lineage label map

Fixed input (`gene_labels_lineage.parquet`). Each Xenium panel gene is mapped to its top lineage by p_top/p_ratio over the seven-lineage reference, with explicit `ambiguous` labels for genes that fail the thresholds. V0 reads this as-is.

In [ ]:
gene_lin = lib.load_lineage_label_map()
import zarr
gene_names = list(zarr.open(lib.TX_ZARR, mode="r").attrs["gene_names"])
print(f"panel size: {len(gene_names)} genes")
print(f"  lineage-mapped (non-ambiguous): {(gene_lin >= 0).sum()}")
for k, L in enumerate(lib.LINEAGES):
    print(f"    {k} {L}: {(gene_lin == k).sum()} genes")

## Stage 3 — Anchor extraction

Iterate the 250 µm transcript grid tiles that overlap the ROI bbox; keep transcripts with `qv ≥ 20`, `valid == 1`, and a lineage-mapped gene. Convert to pixel coordinates in the **crop frame** (origin at the ROI top-left).

In [ ]:
py, px, li = lib.load_anchors_in_bbox(y0, y1, x0, x1, gene_lin)
print(f"anchors in ROI: {len(py):,}")
for k, L in enumerate(lib.LINEAGES):
    n = int((li == k).sum())
    print(f"  {L}: {n:,}")

In [ ]:
from matplotlib.lines import Line2D
fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(np.log1p(s18), cmap="gray")
colors = plt.get_cmap("tab10").colors
for k, L in enumerate(lib.LINEAGES):
    m = li == k
    ax.scatter(px[m], py[m], s=4, color=colors[k], edgecolors="none", alpha=0.7, label=L)
ax.set_xticks([]); ax.set_yticks([]); ax.set_title(f"18S + anchors — {BENCHMARK_ID}")
ax.legend(loc="upper right", fontsize=8, markerscale=2.5)
plt.tight_layout(); plt.show()

## Stage 4 — Lineage posterior (Dirichlet + grey-abstain)

For each lineage *k*: density via isotropic Gaussian (σ = 2 µm = 9.4 px), effective count *N_k* = 2π σ² · ρ_k, total *N* = Σ_k *N_k*.

Dirichlet posterior with α = 10 (high concentration → resists 2-transcript blips). Grey-abstain confidence *c* = *N* / (*N* + N_min) with N_min = 3 blends π toward uniform where anchors are sparse, suppressing spurious edge signal downstream.

In [ ]:
pi_abst, confidence, N_eff = lib.lineage_posterior(py, px, li, H, W)
print(f"pi_abst: shape={pi_abst.shape} range=[{pi_abst.min():.3f}, {pi_abst.max():.3f}]")
print(f"confidence: range=[{confidence.min():.3f}, {confidence.max():.3f}]  mean={confidence.mean():.3f}")

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.ravel()
for k in range(lib.K):
    axes[k].imshow(pi_abst[k], cmap="viridis", vmin=0, vmax=1)
    axes[k].set_title(f"π[{lib.LINEAGES[k]}]")
    axes[k].set_xticks([]); axes[k].set_yticks([])
im = axes[7].imshow(confidence, cmap="magma", vmin=0, vmax=1)
axes[7].set_title("confidence  c = N/(N + N_min)")
axes[7].set_xticks([]); axes[7].set_yticks([])
plt.tight_layout(); plt.show()

## Stage 5 — Edge field (one-vs-rest)

For each lineage *k*, signed margin *s_k* = π_k − max_{j≠k} π_j. The sign of *s_k* changes exactly where lineage *k* transitions from dominant to non-dominant — i.e., at lineage boundaries. We take ‖∇ tanh(K_sharp · s_k)‖ with K_sharp = 8 and sum across *k*. Heterotypic boundaries are detected from both sides and so are emphasised.

In [ ]:
edge_total, top_idx, margin = lib.edge_global_field(pi_abst)
print(f"edge_total: shape={edge_total.shape} max={edge_total.max():.4f}  mean={edge_total.mean():.4f}")
print(f"top_idx values: {sorted(set(top_idx.ravel().tolist()))}")

In [ ]:
# Visualize argmax label map + edge field
from matplotlib.colors import ListedColormap
lin_cmap = ListedColormap([plt.get_cmap("tab10").colors[k] for k in range(lib.K)])
fig, axes = plt.subplots(1, 3, figsize=(16, 5.5))
axes[0].imshow(top_idx, cmap=lin_cmap, vmin=-0.5, vmax=lib.K-0.5)
axes[0].set_title("argmax(π) — lineage label")
axes[1].imshow(margin, cmap="magma", vmin=0, vmax=margin.max())
axes[1].set_title("margin = π_top1 − π_top2")
axes[2].imshow(edge_total, cmap="hot", vmin=0, vmax=np.percentile(edge_total, 99.5))
axes[2].set_title("edge_total = Σ_k |∇ tanh(K · s_k)|")
for a in axes: a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()

## Stage 6 — Cell-evidence gate (morphology only)

Restrict the cut to cellular regions using `max(DAPI_n, 18S_n)`, where both channels are 1-99 percentile-clipped to [0, 1] using the cached **WSI** percentiles. This excludes pure-background pixels without any reference to a prior segmentation.

In [ ]:
percentiles = lib.load_wsi_percentiles()
print(f"WSI cached percentiles: DAPI=[{percentiles['DAPI']['q_lo']}, {percentiles['DAPI']['q_hi']}]  18S=[{percentiles['18S']['q_lo']}, {percentiles['18S']['q_hi']}]")
evidence = lib.cell_evidence(dapi, s18, percentiles)
print(f"cell_evidence: shape={evidence.shape} range=[{evidence.min():.3f}, {evidence.max():.3f}]  mean={evidence.mean():.3f}")
plt.figure(figsize=(6, 6))
plt.imshow(evidence, cmap="gray", vmin=0, vmax=1)
plt.title("cell_evidence = max(DAPI_n, 18S_n)"); plt.xticks([]); plt.yticks([])
plt.tight_layout(); plt.show()

## Stage 7 — Cut field

Weighted edge: *E** = edge_total · *c*² · evidence. Then Gaussian smooth (σ_cut = 5 px) and rescale so the maximum equals depth_max = 0.99.

In [ ]:
cut, weighted_edge = lib.cut_field_from_edge(edge_total, confidence, evidence)
print(f"weighted_edge: max={weighted_edge.max():.4f}  mean={weighted_edge.mean():.4f}")
print(f"cut field:     max={cut.max():.4f}  mean={cut.mean():.4f}")
fig, axes = plt.subplots(1, 3, figsize=(16, 5.5))
axes[0].imshow(weighted_edge, cmap="hot", vmin=0, vmax=np.percentile(weighted_edge, 99.5))
axes[0].set_title("weighted_edge = edge · c² · evidence")
axes[1].imshow(cut, cmap="hot", vmin=0, vmax=1)
axes[1].set_title(f"cut_field (σ={lib.SIGMA_CUT_PX}px, depth={lib.DEPTH_MAX})")
axes[2].imshow(np.log1p(s18), cmap="gray")
axes[2].contour((cut > 0.5).astype(int), levels=[0.5], colors="cyan", linewidths=0.8)
axes[2].set_title("18S + cut > 0.5 contour")
for a in axes: a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()

## Stage 8 — Apply cut → modified 18S

`s_cut = round(S18_raw_uint16 · (1 − cut_field))`. The depth_max cap of 0.99 means even the strongest cuts retain 1% of the 18S signal (avoid pure-zero pixels that confuse CP-SAM normalization).

In [ ]:
s_cut = lib.apply_cut(s18, cut)
fig, axes = plt.subplots(1, 2, figsize=(11, 5.5))
axes[0].imshow(np.log1p(s18),   cmap="gray"); axes[0].set_title("raw 18S")
axes[1].imshow(np.log1p(s_cut), cmap="gray"); axes[1].set_title("s_cut = 18S × (1 − cut)")
for a in axes: a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()

## Stage 9 — Cellpose-SAM segmentation

Run Cellpose-SAM twice with identical settings — once on (DAPI, raw 18S), once on (DAPI, s_cut). Each channel is 1st-99th percentile-clipped to [0, 1] on its own data before the model call.

In [ ]:
print("running CP-SAM on (DAPI, raw 18S) — control…")
masks_ctrl, info_ctrl = lib.run_cpsam(dapi, s18)
print(f"  n_cells = {info_ctrl['n_cells']}")

print("\nrunning CP-SAM on (DAPI, s_cut) — treated…")
masks_cut, info_cut = lib.run_cpsam(dapi, s_cut)
print(f"  n_cells = {info_cut['n_cells']}")

## Stage 10 — Visual comparison

Side-by-side rendering: 18S + mask overlay (control) vs s_cut + mask overlay (treated). Each mask boundary in white.

In [ ]:
def label_overlay(mask, alpha=0.45):
    u = np.unique(mask); u = u[u != 0]
    cm = plt.get_cmap("tab20")(np.linspace(0, 1, max(len(u), 1)))
    out = np.zeros((*mask.shape, 4), dtype=np.float32)
    for i, L in enumerate(u): out[mask == L] = (*cm[i % len(cm)][:3], alpha)
    return out

fig, axes = plt.subplots(1, 2, figsize=(13, 6.5))
axes[0].imshow(np.log1p(s18),   cmap="gray")
axes[0].imshow(label_overlay(masks_ctrl))
b = find_boundaries(masks_ctrl, mode="inner")
axes[0].contour(b.astype(int), levels=[0.5], colors="white", linewidths=0.5)
axes[0].set_title(f"Control: CP-SAM on (DAPI, raw 18S)  →  n_cells = {info_ctrl['n_cells']}")

axes[1].imshow(np.log1p(s_cut), cmap="gray")
axes[1].imshow(label_overlay(masks_cut))
b = find_boundaries(masks_cut, mode="inner")
axes[1].contour(b.astype(int), levels=[0.5], colors="white", linewidths=0.5)
axes[1].set_title(f"Treated: CP-SAM on (DAPI, s_cut)  →  n_cells = {info_cut['n_cells']}")

for a in axes: a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()

In [ ]:
# Focal-cell zoom: locate the doublet's WSI cell ID in both masks and check the count
# inside its bookmark footprint. (For evaluation only — the cut field never used this.)
cps_focal = int(row["cps_id_at_bookmark"])
# Load WSI mask at the bookmark cell to recover its footprint
with tifffile.TiffFile(lib.DATA / "cpsam_whole_slide" / "masks.tif") as tf:
    wsi_mask_crop = tf.series[0].asarray()[y0:y1, x0:x1]
focal_region = (wsi_mask_crop == cps_focal)
print(f"focal cell {cps_focal} footprint in crop: {focal_region.sum()} px")

def count_focal(m, focal):
    a = focal.sum()
    if a == 0: return 0, []
    lbls = np.unique(m[focal]); lbls = lbls[lbls != 0]
    covers = sorted([(int(L), int(((m==L)&focal).sum())/a) for L in lbls], key=lambda kv:-kv[1])
    return sum(1 for _, c in covers if c >= 0.05), covers[:5]

n_ctrl, cov_ctrl = count_focal(masks_ctrl, focal_region)
n_cut,  cov_cut  = count_focal(masks_cut,  focal_region)
print(f"\nFocal-cell split count:")
print(f"  control: n_focal = {n_ctrl}   coverage: {[f'{c*100:.0f}%' for _, c in cov_ctrl[:3]]}")
print(f"  treated: n_focal = {n_cut}   coverage: {[f'{c*100:.0f}%' for _, c in cov_cut[:3]]}")
delta = "+" if n_cut > n_ctrl else ("=" if n_cut == n_ctrl else "−")
print(f"  → V0 cut {'split' if delta == '+' else 'preserved' if delta == '=' else 'merged'} the doublet ({delta})")

---
## V0 pipeline summary

| Stage | Output | Role |
|---|---|---|
| 1 ROI crop | `dapi, s18` (uint16) | 2048×2048 px input |
| 2 Gene labels | `gene_lin` (int8, length panel) | Fixed lineage assignment |
| 3 Anchors | `py, px, li` | qv≥20, lineage-mapped transcripts |
| 4 Posterior | `pi_abst, confidence, N_eff` | Per-pixel lineage probability |
| 5 Edge field | `edge_total, top_idx, margin` | One-vs-rest sign-change detector |
| 6 Cell-evidence | `evidence` | Morphology-only gate |
| 7 Cut field | `cut` | Smoothed, depth-clipped attenuation map |
| 8 Apply cut | `s_cut` (uint16) | Modified 18S |
| 9 CP-SAM | `masks_ctrl, masks_cut` | Two segmentations to compare |

All numerical constants and the math live in `lib.py`. Companion notebook `01_batch_dev_doublets.ipynb` runs this same pipeline across the 68 dev doublets and produces summary figures.